# 🌿 Agentic RAG Interactive Playbook: Step-by-Step Tutorial
### Grounded AI Tutor over *preuniversity.grkraj.org* (12 Chapters)
**Stack**: LangChain • LangGraph • ChromaDB • BM25 • Ollama • Python

---

## 💡 What You Will Learn in This Playbook
1. **Ingestion & Preprocessing**: How raw educational chapters are parsed and cleaned.
2. **Chunking & Metadata Preservation**: Splitting text while preserving Chapter and Section titles.
3. **Dense Vector Embeddings (ChromaDB)**: Transforming text into semantic embeddings.
4. **Sparse Keyword Search (BM25)**: Indexing exact biological terminology.
5. **Hybrid Search & Reciprocal Rank Fusion (RRF)**: Merging dense and sparse results.
6. **LangGraph Agentic State Machine**: How relevance grading, query rewriting, grounded generation, and anti-hallucination checks work step-by-step.
7. **Evaluation & Stress Testing**: Verifying factual accuracy and clean refusal on out-of-scope queries.

## 0. Setup & Environment Verification
Let's make sure our workspace paths and dependencies are correctly loaded.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on Python path
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CORPUS_DIR, CHROMA_DIR, OLLAMA_MODEL, EMBEDDING_MODEL_NAME
print(f"[*] Project Root: {PROJECT_ROOT}")
print(f"[*] Active Ollama Model: {OLLAMA_MODEL}")
print(f"[*] Embeddings Model: {EMBEDDING_MODEL_NAME}")
print(f"[*] Corpus Directory: {CORPUS_DIR}")

---
## Step 1: Loading & Inspecting the 12 Chapters
We load all 12 chapters scraped from `preuniversity.grkraj.org` and inspect their metadata.

In [ ]:
from src.ingestion import load_documents

raw_docs = load_documents()
print(f"[OK] Loaded {len(raw_docs)} full chapter documents:\n")
for i, doc in enumerate(raw_docs):
    print(f"  {i+1:02d}. {doc.metadata['source']} -> '{doc.metadata['chapter_title']}' ({len(doc.page_content)} characters)")

---
## Step 2: Intelligent Chunking & Metadata Enrichment
Why not just embed the whole chapter? Large documents dilute semantic vectors.
We use `RecursiveCharacterTextSplitter` (chunk_size=800, overlap=150) and extract immediate section headings (`section_title`) for every chunk.

In [ ]:
from src.ingestion import chunk_documents

chunks = chunk_documents(raw_docs)
print(f"[OK] Total Chunks Created: {len(chunks)}\n")

# Inspect a sample chunk
sample = chunks[0]
print("--- SAMPLE CHUNK PREVIEW ---")
print(f"Source File:    {sample.metadata['source']}")
print(f"Chapter Title:  {sample.metadata['chapter_title']}")
print(f"Section Title:  {sample.metadata['section_title']}")
print(f"Chunk ID:       {sample.metadata['chunk_id']}")
print("\nContent:\n", sample.page_content)

---
## Step 3 & 4: Dense Vector Index (ChromaDB) + Sparse Index (BM25)
- **Dense Vector Search**: Understands conceptual meaning (e.g. "how plants drink water" matches "transpiration pull").
- **Sparse BM25 Search**: Matches exact keywords and scientific nomenclature (e.g. "RuBisCO", "Kranz anatomy", "pachynema").

In [ ]:
from src.hybrid_retriever import HybridRetriever

retriever = HybridRetriever()
print("[OK] Hybrid Retriever loaded successfully with persistent ChromaDB and BM25 index!")

---
## Step 5: Side-by-Side Comparison of Dense vs. Sparse vs. Hybrid Fusion
Let's test a query to see why **Hybrid Search with Reciprocal Rank Fusion (RRF)** beats either method alone!

In [ ]:
test_query = "How do potassium ions regulate stomata aperture in guard cells?"

dense_results = retriever.dense_search(test_query, k=3)
sparse_results = retriever.sparse_search(test_query, k=3)
hybrid_results = retriever.hybrid_retrieve(test_query, k_final=3)

print(f"QUERY: '{test_query}'\n")
print("=== 1. DENSE VECTOR MATCHES (ChromaDB) ===")
for i, (doc, score) in enumerate(dense_results):
    print(f"  [{i+1}] Score: {score:.3f} | {doc.metadata['source']} ({doc.metadata['section_title']})")

print("\n=== 2. SPARSE KEYWORD MATCHES (BM25) ===")
for i, (doc, score) in enumerate(sparse_results):
    print(f"  [{i+1}] Score: {score:.3f} | {doc.metadata['source']} ({doc.metadata['section_title']})")

print("\n=== 3. FUSED HYBRID MATCHES (RRF) ===")
for i, doc in enumerate(hybrid_results):
    print(f"  [{i+1}] RRF Score: {doc.metadata['fusion_score']} | {doc.metadata['source']} ({doc.metadata['section_title']})")

---
## Step 6: LangGraph Agentic Pipeline Workflow
Traditional RAG blindly passes retrieved documents to the LLM. **Agentic RAG with LangGraph** uses a state machine with guardrails:

1. `retrieve`: Runs hybrid search
2. `grade_documents`: Verifies if retrieved context is actually relevant
3. `transform_query`: Rewrites vague queries if grading fails
4. `generate`: Generates grounded pedagogical answers with `[Chapter X: Section Y]` citations
5. `check_hallucination`: Ensures no ungrounded claims are hallucinated
6. `refuse`: Clean, deterministic refusal path for out-of-scope questions

In [ ]:
from src.agent_graph import build_rag_graph

app = build_rag_graph()
print("[OK] LangGraph StateGraph compiled!")

---
## Step 7: Testing Factual Queries with Citations
Let's execute a real curriculum question across multiple chapters.

In [ ]:
from src.agent_graph import ask_question

q1 = "Explain the structural features of the Watson-Crick B-DNA double helix model."
print(f"Asking: '{q1}'...\n")

result1 = ask_question(q1)
print("=== GENERATED ANSWER ===")
print(result1["generation"])

print("\n=== VERIFIED CITATIONS ===")
for c in result1["citations"]:
    print(f"  • {c['source']} [Section: {c['section']}]")
    print(f"    Excerpt: '{c['snippet']}'\n")

---
## Step 8: Testing the Refusal Path (Anti-Hallucination Guardrail)
What happens if a user asks a non-biology or out-of-domain question (e.g. computer science or stock market)?
Our LangGraph pipeline grades the context as irrelevant and triggers the deterministic refusal path.

In [ ]:
q_adversarial = "What was the closing stock price of Apple on NASDAQ yesterday?"
print(f"Asking out-of-domain query: '{q_adversarial}'...\n")

result_adv = ask_question(q_adversarial)
print("=== ASSISTANT RESPONSE ===")
print(result_adv["generation"])
print(f"Citations returned: {len(result_adv['citations'])}")

---
## Step 9: Running the Automated 15-Question Benchmark
Let's run a quick 3-question mini-evaluation testing direct fact retrieval, cross-chapter reasoning, and refusal.

In [ ]:
from src.eval_suite import EVAL_BENCHMARK_DATA
import time

# Run first query from each category
sample_eval_queries = [EVAL_BENCHMARK_DATA[0], EVAL_BENCHMARK_DATA[5], EVAL_BENCHMARK_DATA[10]]

print("=== RUNNING BENCHMARK MINI-SUITE ===")
for item in sample_eval_queries:
    q = item["question"]
    cat = item["category"]
    t0 = time.time()
    res = ask_question(q)
    latency = round(time.time() - t0, 2)
    
    is_refusal = "cannot find sufficient information" in res["generation"].lower()
    status = "PASS" if ((item["expected_type"] == "refusal" and is_refusal) or (item["expected_type"] == "answer" and not is_refusal)) else "FAIL"
    
    print(f"\n[{cat}] Query: {q}")
    print(f"  → Latency: {latency}s | Status: {status} | Citations: {len(res['citations'])}")
    print(f"  → Snippet: {res['generation'][:150]}...")

---
## 🎉 Summary: Key RAG Best Practices Demonstrated
1. **Never skip metadata**: Chunking with Chapter and Section titles enables precise citations.
2. **Hybrid Retrieval is essential**: BM25 handles specialized terminology; Chroma handles natural language semantics.
3. **Stateful Graph Guardrails**: Grading document relevance before generation prevents hallucinations on out-of-scope queries.
4. **Deterministic Refusal**: A high-quality RAG app knows when to say *"I don't know"*.